In [ ]:
import pickle

# Load pickle file
with open("pt_decoding_data_S62_zscore.pkl", "rb") as f:
    data = pickle.load(f)

In [ ]:
# This data has the following patients:
data.keys()

In [ ]:
# Each person's data has the following keys:
data['S14'].keys()

# PCA & SVM

In [ ]:
# Core math & data handling
import numpy as np

# Model building
from sklearn.decomposition import PCA
from sklearn.svm import SVC
from sklearn.pipeline import make_pipeline

# Model evaluation and splitting
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

# Hyperparameter search
from skopt import BayesSearchCV   # pip install scikit-optimize

In [ ]:
# This function actually executes the PCA_SVM algorithm

def PCA_SVM(X, y):
# X: brain signal (trials by time), 
# y: phoneme labels (trials)

    # STEP 1: split samples into train & test set, based on each sample's target output
    # in total 144 samples
    idx = np.arange(len(X))
    
    train_idx, test_idx = train_test_split(
        idx,
        stratify=y, # if my data is 30% phoneme A and 70% phoneme B, it keeps both train & test at 30%/70%
        test_size=0.2, # 20% for test, 80% for train
        random_state=0 # RNG seed is fixed, reproducible
    )

    X_train, X_test = X[train_idx], X[test_idx]
    y_train, y_test = y[train_idx], y[test_idx]

    # STEP 2: establish the full pipeline (input X_train & y_train, output division lines)
    # clf: the full pipeline for PCA + SVC
    clf = make_pipeline(
        PCA(),
        SVC(kernel='rbf', class_weight='balanced') 
        # rbf = nonlinear Gaussian boundary, 
        # class_weight='balanced' -> if some phonemes are rarer, upweigh their errors
    )

    # STEP 3: guess 3 key model params that will influence training performance
    # search: a bayes optimizer object that can 
    # 1) guess params that would produce the most accurate classification (we use this),
    # 2) actually use those params to do the classification (we don't use this)
    search = BayesSearchCV(
        clf,
        {
            # params to be guessed:
            'pca__n_components': (0.1, 0.95, 'uniform'),
                # each input sample is an array of 22,200 numbers. 
                # PCA creates new components that are a weighted combination of the 22,200 numbers. 
                # pca__n_components = x%, pick the smallest number of components that can explain x% of variance
            'svc__C': (1e-3, 1e5, 'log-uniform'),
                # C: how harshly to penalize misclassifications
            'svc__gamma': (1e-4, 1e3, 'log-uniform'),
                # gamma: how wiggly the decision boundary is
        },
        cv=5,  
        # 5-fold cross validation: score each param set by 
        # 1) split the training set into 5 shares, 
        # 2) produce 5 scores using each share as validation, 
        # 3) avg the 5 scores.          
        n_iter=25, 
        # try 25 different param sets   
        n_points=5,    
        # evaluate up to 5 param sets in parallel
        n_jobs=-1,     
        # use all CPU cores
        refit=False,   
        # don't automatically start fitting
        verbose=1,      
        # print progress
        random_state=0
        # no randomness
    )

    # STEP 4: actually start guessing the params
    # Run the search on training data
    search.fit(X_train, y_train)

    # STEP 5: run the model on train set
    # Now that seach.fit has finished running,
    # the best params are ready
    # use that best params set on the actual training set to get the model trained
    clf.set_params(**search.best_params_)
    clf.fit(X_train, y_train)

    # STEP 6: run the model on test set
    # now that clf.fit has finished running,
    # we have the model ready
    # output the predicted phoneme labels by running this model on the test set
    y_pred = clf.predict(X_test)

    return y_test, y_pred


In [ ]:
# This function reports accuracy after obtaining predictions

def report_accuracy(y_test, y_pred):

    accuracy = float(np.mean(y_test == y_pred))

    print(f'The accuracy is {accuracy}')

    return accuracy
    

In [ ]:
# X1 = data['S14']['X1']
# X1_flattened = data['S14']['X1'].reshape(X1.shape[0], X1.shape[1]*X1.shape[2])
# phon_labels = data['S14']['y_full_phon']
# phon_1_labels = phon_labels[:,0]

# X = X1_flattened
# y = phon_1_labels

# y_test, y_pred = PCA_SVM(X, y)
# accuracy = report_accuracy(y_test, y_pred)

# This function runs the PCA_SVM algorithm on selected patients & phonemes 

def PCA_SVM_batch(patient_names, phoneme_labels, data):
# patient_names: which patients to run the alg on, e.g. ['S14', 'S26']
# phoneme_labels: which phonemes to run the alg on, e.g. ['X1', 'X2']
# data: raw, full data

    results = []  # to store (patient, phoneme, accuracy)

    for patient in patient_names:
        for phon in phoneme_labels:
            # Load the feature matrix for this patient+phoneme
            X = data[patient][phon]
            # trail × time × channels -> trial × (time × channels)
            X_flattened = X.reshape(X.shape[0], -1)

            # Labels: full phoneme label matrix [n_trials, n_phonemes]
            phon_labels = data[patient]['y_full_phon']
            # y: [n_trials], a single col of the target phon
            phon_idx = int(phon[1:]) - 1  
            y = phon_labels[:, phon_idx]

            # Run PCA+SVM pipeline
            y_test, y_pred = PCA_SVM(X_flattened, y)

            # Compute accuracy
            acc = report_accuracy(y_test, y_pred)

            # Save result
            results.append((patient, phon, acc))

    # Pretty print report
    print("=== PCA+SVM Batch Results ===")
    for patient in patient_names:
        print(f"Patient {patient}:")
        for phon in phoneme_labels:
            acc = [r[2] for r in results if r[0] == patient and r[1] == phon][0]
            print(f"  Phoneme {phon}: {acc:.3f}")

    return results

r = PCA_SVM_batch(['S14', 'S26', 'S23', 'S33', 'S22', 'S39', 'S58', 'S62'], ['X1', 'X2', 'X3'], data)

In [ ]:
unique_phon = len(np.unique(data['S14']['y_phon_collapsed']))
print(f'There are {unique_phon} unique phonemes. Baseline chance of guessing correctly is {1/unique_phon}.')

# View Results

In [ ]:
def plot_patient_phoneme_bars(r):
    """
    Plot grouped bar chart for patient accuracies with baseline line.
    
    Parameters:
        r (list of tuples): Each tuple should be (Patient, Phoneme, Accuracy)
    """
    import pandas as pd
    import matplotlib.pyplot as plt

    # convert to DataFrame
    df = pd.DataFrame(r, columns=["Patient", "Phoneme", "Accuracy"])
    pivot_df = df.pivot(index="Patient", columns="Phoneme", values="Accuracy")

    # plot
    ax = pivot_df.plot(kind="bar", figsize=(8,5))
    plt.title("Accuracy by Patient and Phoneme")
    plt.ylabel("Accuracy")
    plt.xlabel("Patient")
    plt.xticks(rotation=45)
    plt.legend(title="Phoneme")

    # add baseline chance line
    baseline = 0.1111111111111111
    plt.axhline(y=baseline, color='red', linestyle='dotted', linewidth=2, label="Baseline Chance")

    # update legend
    handles, labels = ax.get_legend_handles_labels()
    ax.legend(handles, labels, title="Phoneme / Baseline")

    plt.tight_layout()
    plt.show()


# test the function with given data
plot_patient_phoneme_bars(r)
